In [ ]:
import wrds
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import getpass

# --- Visualization Settings ---
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
print("===== Investment Guidance for Beginners =====")
print("1. Higher volatility represents higher investment risk.")
print("2. Beta > 1 means the stock is more volatile than the overall market.")
print("3. Lower maximum drawdown indicates weaker crash risk.")
print("4. Higher Sharpe ratio means better risk-adjusted return.")
print("5. This tool helps student investors evaluate stocks in a simple way.\n")

In [ ]:
# --- 1. Database Connection ---
print("===== WRDS Database Login =====")
print("Please enter your WRDS credentials.")
username = input("Enter your WRDS username: ")
password = getpass.getpass("Enter your WRDS password: ")

try:
    db = wrds.Connection(username=username, password=password)
    print("Connection successful!\n")
except Exception as e:
    print(f"Connection failed: {e}")
    # Stop execution if connection fails
    raise SystemExit("Stopping script due to connection failure.")

In [ ]:
# --- 2. User Input & Configuration ---
portfolio = {} # Renamed from stock_dict for clarity

print("===== Custom Stock Risk Analysis System =====")
print("Example Permno: 14593 (Apple), 93536 (Tesla)\n")

while True:
    name = input("Please enter Stock Name: ")
    permno = input("Enter Stock Permnor: ")
    portfolio[name] = permno.upper()
    
    if input("Add another stock? (y / n): ").lower() == "n":
        break

# Allow user to define date range (Best Practice)
start_date = input("Enter Start Date (YYYY-MM-DD) [default 2015-01-01]: ") or "2015-01-01"
end_date = input("Enter End Date (YYYY-MM-DD) [default 2025-01-01]: ") or "2025-01-01"

In [ ]:
# --- 3. Data Extraction Functions ---
def get_wrds_stock_data(permno, start, end):
    sql_query = f"""
    SELECT date, prc, ret
    FROM crsp.dsf
    WHERE permno = {permno}
    AND date >= '{start}' AND date <= '{end}'
    ORDER BY date
    """
    df = db.raw_sql(sql_query)
    if df.empty:
        print(f"Warning: No data found for permno: {permno}")
        return df
    
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df["prc"] = df["prc"].abs() # Handle negative prices in CRSP
    return df

In [ ]:
# --- 4. Download Data ---
stock_data_collection = {}

# A. Download Stocks
for name, permno in portfolio.items():
    print(f"Downloading data: {name} ({permno})...")
    stock_data_collection[name] = get_wrds_stock_data(permno, start_date, end_date)

# B. Download Market Data (S&P 500)
print("Downloading S&P 500 Market Data...")
market_sql = f"""
SELECT date, vwretd
FROM crsp.dsi
WHERE date >= '{start_date}' AND date <= '{end_date}'
ORDER BY date
"""
market_df = db.raw_sql(market_sql)
market_df["date"] = pd.to_datetime(market_df["date"])
market_df = market_df.set_index("date")

# Generate Market Index Price for Plotting
market_df["market_index_price"] = (1 + market_df["vwretd"]).cumprod()
market_return = market_df["vwretd"]

In [ ]:
# --- 5. Calculation Functions ---
def calculate_volatility(return_series):
    return return_series.dropna().std() * np.sqrt(252)

def calculate_beta(stock_return, market_return):
    # Align data first
    combined_data = pd.concat([stock_return, market_return], axis=1).dropna()
    if combined_data.empty:
        return np.nan
    covariance = combined_data.cov().iloc[0, 1]
    # FIX: Use variance of the aligned market data
    market_variance = combined_data.iloc[:, 1].var() 
    if market_variance == 0:
        return np.nan
    return covariance / market_variance

def calculate_max_drawdown(price_series):
    rolling_max = price_series.cummax()
    drawdown = (price_series - rolling_max) / rolling_max
    return drawdown.min()

def calculate_sharpe(return_series, risk_free_rate=0.02):
    clean_return = return_series.dropna()
    if clean_return.empty or clean_return.std() == 0:
        return np.nan
    annual_return = clean_return.mean() * 252
    volatility = clean_return.std() * np.sqrt(252)
    return (annual_return - risk_free_rate) / volatility

In [ ]:
# --- 6. Perform Analysis ---
result_list = []

# A. Analyze Individual Stocks
for name, df in stock_data_collection.items():
    # FIX: Skip if data is empty (e.g., wrong permno)
    if df.empty:
        continue
        
    daily_return = df["ret"]
    stock_price = df["prc"]
    
    vol = round(calculate_volatility(daily_return), 4)
    beta = round(calculate_beta(daily_return, market_return), 4)
    max_draw = round(calculate_max_drawdown(stock_price), 4)
    sharpe = round(calculate_sharpe(daily_return), 4)
    
    result_list.append({
        "Stock Name": name,
        "Annualized Volatility": vol,
        "Beta Coefficient": beta,
        "Maximum Drawdown": max_draw,
        "Sharpe Ratio": sharpe
    })

# B. Analyze Market Index (To be included in the table and bar chart)
# We treat the market index as a "stock" here for comparison
market_vol = round(calculate_volatility(market_return), 4)
# Market Beta is always 1 relative to itself
market_beta = 1.0 
market_max_draw = round(calculate_max_drawdown(market_df["market_index_price"]), 4)
market_sharpe = round(calculate_sharpe(market_return), 4)

result_list.append({
    "Stock Name": "S&P 500 (Market)",
    "Annualized Volatility": market_vol,
    "Beta Coefficient": market_beta,
    "Maximum Drawdown": market_max_draw,
    "Sharpe Ratio": market_sharpe
})

# Generate Result Table
result_df = pd.DataFrame(result_list)
print("\n===== Final Stock Risk Analysis Result =====")
display(result_df)

In [ ]:
# --- 7. Visualization ---

# Prepare Data for Plotting (Add Market to collection)
market_for_plot = pd.DataFrame()
market_for_plot["prc"] = market_df["market_index_price"]
stock_data_collection["S&P 500 (Market)"] = market_for_plot

# Chart 1: Normalized Price Trend
plt.figure(figsize=(12, 6))
for name, df in stock_data_collection.items():
    if df.empty: continue
    normalized_price = df["prc"] / df["prc"].iloc[0]
    plt.plot(normalized_price, label=name, linewidth=2)

plt.title("Normalized Stock Price Trend Comparison")
plt.ylabel("Normalized Price")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Chart 2: Volatility Bar Chart (Now includes Market)
plt.figure(figsize=(10, 5))
# Use the result_df which now contains the market data
plt.bar(result_df["Stock Name"], result_df["Annualized Volatility"], color="#4682B4")
plt.title("Annualized Volatility Comparison")
plt.ylabel("Annualized Volatility")
plt.xticks(rotation=45) # Rotate labels if they are long
plt.show()

In [ ]:
# --- 8. Cleanup ---
db.close()
print("Analysis complete. Database connection closed.")